# ¿Qué cuenta va aquí? — Un asistente de codificación para el auxiliar que llega nuevo a la firma

**Proyecto final · Metodologías y herramientas para la analítica y visualización de datos**
Maestría en Analítica de Datos para Contabilidad y Auditoría · Universidad Externado de Colombia · Cohorte 3

---

### El encargo

Una firma contable lleva la contabilidad de varias empresas en dos softwares distintos. Cada mes su equipo codifica a mano miles de líneas de auxiliar: decidir, para cada movimiento, **a qué cuenta del PUC va**. Quien lleva años en la firma lo hace casi sin pensar. Quien llega nuevo no tiene esa memoria: pregunta, duda, y se equivoca en cosas que el equipo ya resolvió mil veces.

**La pregunta:** ¿puede un auxiliar nuevo codificar como el equipo experimentado desde su primer mes, si un modelo entrenado con la codificación histórica de la firma le sugiere la cuenta y le avisa cuándo debe consultar?

**El destinatario:** el socio de la firma. **El beneficiario:** el auxiliar que entra. **La decisión que se pide:** aprobar o rechazar el uso del asistente como referencia obligatoria en los primeros meses de cada auxiliar, con un protocolo de tres zonas —sigue, elige entre tres, consulta— y una forma de medir si funciona.

Como beneficio secundario, el mismo asistente permite automatizar las líneas en las que tiene alta confianza y reducir horas del equipo; se cuantifica, pero no es el objetivo.

### Cómo está organizado

| Nivel | Pregunta | Sección |
|---|---|---|
| Descriptiva | ¿Qué pasó? | 1 |
| Diagnóstica | ¿Por qué pasó? | 2 |
| Predictiva | ¿Qué va a pasar? | 3 |
| Prescriptiva | ¿Qué hacemos? | 4 |

Cada nivel se construye sobre el anterior: lo que se describe en 1 obliga a las preguntas de 2; lo que se encuentra en 2 decide qué variables y qué modelos entran en 3; y 3 solo sirve si 4 lo convierte en una lista de qué hacer el lunes.

### Cómo ejecutarlo

`Entorno de ejecución → Ejecutar todo`. El notebook descarga los datos por sí mismo. Tarda entre 4 y 7 minutos en Colab. Todos los modelos usan semilla fija: los números son reproducibles.

### Sobre la fuente

Son datos reales de seis empresas de la cartera de la firma, **anonimizados antes de salir de la firma**: empresas y terceros como códigos secuenciales, cifras multiplicadas por una constante, nombres borrados del texto. La sección 1.2 documenta exactamente qué se hizo. La autorización del responsable de la información se adjunta al entregable.

## 0 · Preparación

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time, io, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
import sklearn

SEMILLA = 0
np.random.seed(SEMILLA)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams["figure.figsize"] = (9, 4); plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .3
VERDE, ROJO, GRIS, AZUL = "#2d5a50", "#a8442a", "#8a8a8a", "#3b6ea5"
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| python", sys.version.split()[0])

### 0.1 · Carga de datos

El archivo `libro_comun_anon.csv` está publicado en el repositorio del proyecto. Si la descarga falla (por ejemplo, sin conexión), la celda busca el archivo en la carpeta de trabajo y, en Colab, ofrece subirlo.

In [ ]:
URL_DATOS = "https://raw.githubusercontent.com/jonathanr9329-sketch/proyecto-final-maca/main/libro_comun_anon.csv"
ARCHIVO_LOCAL = "libro_comun_anon.csv"

TIPOS = {"cuenta": str, "cuenta4": str, "objetivo": str, "grupo_puc": str}
def cargar():
    try:
        d = pd.read_csv(URL_DATOS, parse_dates=["fecha"], dtype=TIPOS); print("Datos descargados desde el repositorio."); return d
    except Exception as e:
        print("No se pudo descargar desde la URL:", str(e)[:80])
    try:
        d = pd.read_csv(ARCHIVO_LOCAL, parse_dates=["fecha"], dtype=TIPOS); print("Datos leidos del archivo local."); return d
    except Exception:
        pass
    try:
        from google.colab import files
        print("Suba el archivo libro_comun_anon.csv:"); subido = files.upload()
        nombre = next(iter(subido)); return pd.read_csv(io.BytesIO(subido[nombre]), parse_dates=["fecha"], dtype=TIPOS)
    except ImportError:
        raise SystemExit("No se encontraron los datos. Coloque libro_comun_anon.csv junto al notebook o corrija URL_DATOS.")

lb = cargar()
lb["texto_anon"] = lb["texto_anon"].fillna(""); lb["centro"] = lb["centro"].fillna("")
print(f"{len(lb):,} movimientos x {lb.shape[1]} columnas")
lb.head(3)

---
# 1 · NIVEL DESCRIPTIVO · ¿Qué pasó?

Antes de modelar hay que poder responder de dónde vienen los datos, qué cubren, qué les falta y qué se les hizo. Es el mismo hábito que exige un papel de trabajo: si no se puede describir la fuente, cualquier conclusión es indefendible.

## 1.1 · Ficha de la fuente

In [ ]:
ficha = lb.groupby(["software","empresa"]).agg(
    movimientos=("monto","size"), comprobantes=("comprobante_id","nunique"),
    terceros=("tercero","nunique"), cuentas_4dig=("cuenta4","nunique"),
    desde=("fecha","min"), hasta=("fecha","max"),
    lineas_con_texto=("texto_anon", lambda s: (s.str.len()>3).mean()),
    tipos_documento=("tipo_doc","nunique"))
display(ficha)
print(f"Total: {len(lb):,} movimientos | {lb['comprobante_id'].nunique():,} comprobantes | "
      f"{lb['tercero'].nunique():,} terceros | {lb['empresa'].nunique()} empresas | {lb['software'].nunique()} softwares | "
      f"{lb['objetivo'].nunique()} clases de cuenta | periodo {lb['fecha'].min():%Y-%m} a {lb['fecha'].max():%Y-%m}")

| Campo | Contenido |
|---|---|
| Unidad de análisis | Una **línea** del libro auxiliar (un movimiento débito o crédito contra una cuenta) |
| Origen | Exportación del libro auxiliar de cada software contable (Loggro y Siigo), 2025 y 2026 |
| Variable objetivo | `objetivo`: la cuenta del PUC a **cuatro dígitos** (nivel de cuenta). Las cuentas con menos de 100 movimientos se agrupan en `OTRAS` |
| Faltantes | Ninguno en las variables usadas. El texto libre solo existe donde el software lo exporta (ver 1.1) |
| Duplicados | Ninguno tras el control de calidad (ver 1.2) |
| Lo que NO trae | Usuario que registró, fecha de creación del asiento, ni sector de la empresa |

**Por qué cuatro dígitos.** Las cuentas auxiliares de seis u ocho dígitos son parametrización de cada empresa: `51050601` en una empresa no es lo mismo que en otra. El nivel de cuenta del PUC (`5105`, gastos de personal) sí es comparable entre empresas y entre softwares. Además, reduce el problema de miles de clases a unas sesenta, que es lo que hace posible entrenar un clasificador con estos volúmenes.

## 1.2 · Qué se hizo con los datos antes de llegar aquí

Los datos salen de la firma ya anonimizados. El proceso, reproducible en `01_consolidar_anonimizar.py`, hizo cuatro cosas:

1. **Homologó dos softwares a un libro común.** Cada uno exporta columnas distintas; se llevaron a un formato único: fecha, comprobante, tipo de documento, tercero, cuenta, débito, crédito, texto.
2. **Anonimizó.** Empresas → `E01…E06`. Terceros → `T00001…`, con el mismo código en todas las empresas (permite ver proveedores compartidos). Cifras × una constante fija. En el texto libre, se reemplazó por `<T>` cualquier palabra que fuera parte del nombre de un tercero.
3. **Descartó lo que no es un movimiento.** Líneas con débito y crédito en cero (Siigo exporta el detalle de factura y las líneas de "IVA 0 %" como renglones informativos) y líneas sin fecha.
4. **Control de calidad.** Se detectó, por huella del contenido, un export duplicado bajo otro nombre de archivo y se excluyó. Se corrigió una diferencia de formato en la identificación del tercero entre años (texto en 2025, decimal en 2026) que habría partido cada tercero en dos.

Ese último punto es la respuesta a una pregunta clásica de auditoría: *¿qué haría al recibir de un cliente un archivo con columnas duplicadas?* Documentarlo, corregirlo y dejar el rastro.

## 1.3 · Cómo se ve el libro

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
g = pd.crosstab(lb["grupo_puc"], lb["software"], normalize="columns")
g.T.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20", width=.6, legend=True)
axes[0].set_title("Composicion por grupo PUC, segun software"); axes[0].set_ylabel("proporcion de movimientos")
axes[0].legend(title="grupo", fontsize=7, ncol=2)

vc = lb["objetivo"].value_counts(); acum = vc.cumsum()/len(lb)
axes[1].plot(range(1, len(acum)+1), acum.values, color=VERDE); axes[1].axhline(.8, ls="--", color=GRIS)
axes[1].set_title("Concentracion: cuantas cuentas explican el libro"); axes[1].set_xlabel("cuentas, de mas a menos frecuente"); axes[1].set_ylabel("acumulado")

lpc = lb.groupby("comprobante_id").size()
axes[2].hist(lpc.clip(upper=15), bins=15, color=AZUL); axes[2].set_title("Lineas por comprobante (recortado en 15)"); axes[2].set_xlabel("lineas")
plt.tight_layout(); plt.show()

print(f"Las 5 cuentas mas frecuentes concentran el {acum.iloc[4]:.0%} de los movimientos; las 20 mas frecuentes, el {acum.iloc[19]:.0%}.")
print(f"Un comprobante tipico tiene {lpc.median():.0f} lineas (p90 = {lpc.quantile(.9):.0f}, maximo = {lpc.max()}).")
print("\nLas 10 cuentas mas frecuentes:")
display(pd.DataFrame({"movimientos": vc.head(10), "proporcion": (vc.head(10)/len(lb))}))

**Lectura.** Los grupos 1 (activo), 2 (pasivo) y 5 (gastos) dominan en los dos softwares, pero Loggro tiene un grupo 7 (costos de producción) que Siigo no usa, y Siigo pesa más en ingresos y gastos. Ya en el nivel descriptivo se ve que **las carteras no son intercambiables**. La concentración es alta —veinte cuentas explican cuatro de cada cinco líneas— y eso es lo que hace viable un clasificador; el reto está en la cola larga.

---
# 2 · NIVEL DIAGNÓSTICO · ¿Por qué pasó?

El nivel descriptivo dejó una intuición: si veinte cuentas explican el 80 %, ¿no bastaría una tabla que diga "este tercero va a esta cuenta"? Eso es lo primero que hay que poner a prueba, porque **si una regla simple resuelve el problema, no hay proyecto de machine learning que justificar.**

## 2.1 · Hallazgo 1 — El tercero no determina la cuenta

In [ ]:
tc = lb.groupby(["empresa","tercero","cuenta4"]).size().reset_index(name="n")
tc["total"] = tc.groupby(["empresa","tercero"])["n"].transform("sum")
pureza = tc.sort_values("n", ascending=False).drop_duplicates(["empresa","tercero"]).assign(pureza=lambda d: d.n/d.total)
cuentas_por_tercero = tc.groupby(["empresa","tercero"]).size()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(pureza.pureza, bins=20, color=VERDE); ax[0].set_title("Pureza: proporcion del tercero en su cuenta mas frecuente"); ax[0].set_xlabel("pureza"); ax[0].set_ylabel("terceros")
ax[1].hist(cuentas_por_tercero.clip(upper=12), bins=12, color=ROJO); ax[1].set_title("Cuentas distintas que toca cada tercero (recortado en 12)"); ax[1].set_xlabel("cuentas")
plt.tight_layout(); plt.show()
print(f"Solo el {(pureza.pureza==1).mean():.0%} de los terceros va SIEMPRE a la misma cuenta.")
print(f"La mediana es {cuentas_por_tercero.median():.0f} cuentas distintas por tercero (p90 = {cuentas_por_tercero.quantile(.9):.0f}).")

**Por qué.** Es obvio cuando se piensa como contador: una factura de compra toca el gasto, el IVA descontable, la retención y la cuenta por pagar, todas con el mismo tercero. **La cuenta no la define quién es el tercero sino qué papel juega la línea dentro del comprobante**: si es débito o crédito, si es la línea grande o una pequeña, cuántas líneas la acompañan. Esa observación decide qué variables hay que construir en el nivel predictivo.

## 2.2 · Hallazgo 2 — Reglas progresivas: cuánto vale cada pieza de información

Antes de cualquier modelo, tres reglas que cualquiera podría hacer en Excel, entrenadas con 2025 y probadas sobre 2026. Son el punto de comparación obligado: un modelo que no les gane no sirve.

In [ ]:
tr = lb[lb.anio==2025].copy(); te = lb[lb.anio==2026].copy()
mayoritaria = tr["objetivo"].mode()[0]

# B0: siempre la clase mayoritaria
te["B0"] = mayoritaria
# B1: cada tercero -> su cuenta mas frecuente en 2025 (si es nuevo, la mayoritaria)
r1 = tr.groupby(["empresa","tercero","objetivo"]).size().reset_index(name="n").sort_values("n", ascending=False).drop_duplicates(["empresa","tercero"])
mapa1 = dict(zip(zip(r1.empresa, r1.tercero), r1.objetivo))
te["B1"] = [mapa1.get((e,t), mayoritaria) for e,t in zip(te.empresa, te.tercero)]
# B2: tercero + tipo de documento + naturaleza (debito/credito)
r2 = tr.groupby(["empresa","tercero","tipo_doc","naturaleza","objetivo"]).size().reset_index(name="n").sort_values("n", ascending=False).drop_duplicates(["empresa","tercero","tipo_doc","naturaleza"])
mapa2 = dict(zip(zip(r2.empresa, r2.tercero, r2.tipo_doc, r2.naturaleza), r2.objetivo))
te["B2"] = [mapa2.get(k, mapa1.get((k[0],k[1]), mayoritaria)) for k in zip(te.empresa, te.tercero, te.tipo_doc, te.naturaleza)]

vistos = tr.groupby("empresa")["tercero"].apply(set)
te["tercero_nuevo"] = [t not in vistos.get(e, set()) for e,t in zip(te.empresa, te.tercero)]

reglas = pd.DataFrame({
    "regla": ["B0 · siempre la cuenta mas frecuente", "B1 · cada tercero a su cuenta habitual", "B2 · tercero + tipo de documento + naturaleza"],
    "acierto 2026": [accuracy_score(te.objetivo, te.B0), accuracy_score(te.objetivo, te.B1), accuracy_score(te.objetivo, te.B2)]})
display(reglas.set_index("regla"))
print(f"Movimientos de 2026 cuyo tercero no existia en 2025: {te.tercero_nuevo.mean():.1%}. "
      f"Sobre ellos, B1 acierta {accuracy_score(te.objetivo[te.tercero_nuevo], te.B1[te.tercero_nuevo]):.1%}.")

**Lectura.** Saber quién es el tercero vale unos 30 puntos sobre la apuesta ciega. Añadir **una sola columna —débito o crédito—** junto con el tipo de documento vale otros 20. Confirma el Hallazgo 1: el papel de la línea importa tanto como el tercero.

Y queda expuesto el límite de cualquier tabla de mapeo: **los terceros nuevos**. Ninguna regla puede codificar un proveedor o un cliente que no existía el año anterior, y son una parte relevante de cada año.

## 2.3 · Hallazgo 3 — Los dos softwares traen información casi disjunta

In [ ]:
info = lb.groupby("software").agg(lineas_con_texto=("texto_anon", lambda s: (s.str.len()>3).mean()),
                                  tipos_de_documento=("tipo_doc","nunique"),
                                  manual_conocido=("es_manual", lambda s: (s>=0).mean()))
display(info)
print("Tipos de documento mas frecuentes por software:")
for s, d in lb.groupby("software"):
    print(f"  {s}: {d['tipo_doc'].value_counts().head(6).to_dict()}")

**Lectura.** Loggro exporta un tipo de transacción con **nombres estándar del software** (importado, manual, factura de compra, pago…) que además permite saber **qué asiento es manual** — el corazón de las pruebas sobre asientos de diario de la NIA 240 — pero no trae texto libre salvo el que el usuario escribe en los asientos manuales. Siigo trae texto en casi todas las líneas, pero su tipo de comprobante es un **código de parametrización de cada empresa** (`CC-17`, `FV-2`, `RP-1`): el mismo código no significa lo mismo en dos empresas, y no dice si el asiento fue manual. Cada sistema tiene lo que al otro le falta. Eso obliga a un diseño en dos capas: **un modelo con lo común a los dos** y una medición de **cuánto aporta lo propio de cada uno**.

## 2.4 · Hallazgo 4 — Deriva entre años

In [ ]:
d25 = lb[lb.anio==2025].groupby("empresa")["cuenta4"].apply(set); d26 = lb[lb.anio==2026].groupby("empresa")["cuenta4"].apply(set)
deriva = pd.DataFrame({"cuentas_2025": d25.map(len), "cuentas_2026": d26.map(len),
                       "aparecen_en_2026": [len(d26[e]-d25[e]) for e in d25.index], "desaparecen_en_2026": [len(d25[e]-d26[e]) for e in d25.index]})
display(deriva)
nuevos_emp = te.groupby(["software","empresa"])["tercero_nuevo"].mean()
fig, ax = plt.subplots(); nuevos_emp.reset_index(level=0, drop=True).plot(kind="bar", ax=ax, color=ROJO)
ax.set_title("Movimientos de 2026 con terceros que no existian en 2025"); ax.set_ylabel("proporcion"); plt.show()

**Lectura.** Las contabilidades cambian de un año a otro: cuentas que dejan de usarse, cuentas que aparecen, terceros nuevos en proporciones muy distintas según el negocio. Un modelo entrenado con el pasado se va a encontrar con esto. **Por eso la partición del nivel predictivo es temporal —entrenar con 2025, probar con 2026— y no aleatoria:** hay que medir el modelo en las condiciones en que se va a usar.

## 2.5 · Lo que el diagnóstico decide para el modelo

1. **Variables de contexto del comprobante**, no solo del tercero: número de líneas, proporción que representa esta línea del total, si es la línea mayor, cuántos débitos y créditos tiene el documento.
2. **El texto entra como variable** (vectorizado), y se mide aparte cuánto aporta, porque solo existe en parte del libro.
3. **Partición temporal** 2025 → 2026, que además evita la fuga de datos por comprobante: las líneas de un mismo documento comparten tercero, fecha y monto, y si se repartieran al azar el modelo vería casi la misma información a los dos lados.
4. **El baseline B2 (regla tercero + tipo + naturaleza) es el número que hay que superar.**

---
# 3 · NIVEL PREDICTIVO · ¿Qué va a pasar?

## 3.1 · Variables de contexto del comprobante

Se construyen a partir del propio libro; no requieren información externa.

In [ ]:
g = lb.groupby("comprobante_id")
lb["n_lineas"]       = g["monto"].transform("size")
lb["total_cbte"]     = g["monto"].transform("sum")
lb["prop_del_cbte"]  = (lb["monto"] / lb["total_cbte"].replace(0, np.nan)).fillna(0)
lb["es_linea_mayor"] = (lb["monto"] == g["monto"].transform("max")).astype(int)
lb["n_debitos"]      = g["naturaleza"].transform(lambda s: (s=="D").sum())
lb["n_creditos"]     = lb["n_lineas"] - lb["n_debitos"]
lb["tercero_freq"]   = lb.groupby(["empresa","tercero"])["monto"].transform("size")   # que tan habitual es el tercero
lb["tercero_pj"]     = lb["tipo_persona"].fillna("").str.contains("Jur").astype(int)

CATEGORICAS = ["empresa","software","naturaleza","tipo_doc","centro"]
NUMERICAS   = ["log_monto","monto_redondo","mes","dia_semana","fin_de_mes","cierre_dic",
               "n_lineas","prop_del_cbte","es_linea_mayor","n_debitos","n_creditos","tercero_freq","tercero_pj"]

tr = lb[lb.anio==2025].copy(); te = lb[lb.anio==2026].copy()
y_tr, y_te = tr["objetivo"], te["objetivo"]
te["tercero_nuevo"] = [t not in vistos.get(e, set()) for e,t in zip(te.empresa, te.tercero)]
te["B2"] = [mapa2.get(k, mapa1.get((k[0],k[1]), mayoritaria)) for k in zip(te.empresa, te.tercero, te.tipo_doc, te.naturaleza)]
print(f"Entrenamiento (2025): {len(tr):,} lineas | Prueba (2026): {len(te):,} lineas | Clases: {y_tr.nunique()}")

## 3.2 · Cómo se representan los datos y cómo se evalúa

- Las categóricas (empresa, software, naturaleza, tipo de documento, centro de costo y, opcionalmente, el **tercero**) se convierten en columnas 0/1 (*one-hot*). Un valor que aparece en 2026 y no existía en 2025 simplemente no activa ninguna columna: es lo que le pasaría al modelo en producción.
- Las numéricas se estandarizan **con la media y desviación de 2025 solamente**. Escalar con todo el conjunto sería fuga de datos.
- El texto se vectoriza con **TF-IDF**: cada palabra (y cada par de palabras) se vuelve una columna cuyo valor es alto si la palabra es frecuente en esa línea y rara en el resto del libro. Es, en términos contables, darle más peso a la palabra que *discrimina* ("arrendamiento") que a la que está en todas partes ("pago").
- Métricas: **acierto** (proporción de líneas bien codificadas), **F1 macro** (promedio del F1 de cada cuenta, para que las cuentas raras pesen igual que las frecuentes) y **top-3** (si la cuenta correcta está entre las tres que el modelo considera más probables, que es lo que importa para codificación *asistida*).
- Además, para cada umbral de confianza: qué proporción de líneas lo supera y con qué acierto. Ese par de números es la materia prima del nivel prescriptivo.

In [ ]:
def matrices(tr, te, con_tercero=True, con_texto=True):
    cats = CATEGORICAS + (["tercero"] if con_tercero else [])
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=10, dtype=np.float32)
    sc  = StandardScaler()
    Xtr = [ohe.fit_transform(tr[cats].astype(str)), csr_matrix(sc.fit_transform(tr[NUMERICAS]).astype(np.float32))]
    Xte = [ohe.transform(te[cats].astype(str)),     csr_matrix(sc.transform(te[NUMERICAS]).astype(np.float32))]
    nombres = list(ohe.get_feature_names_out(cats)) + NUMERICAS
    if con_texto:
        tf = TfidfVectorizer(min_df=5, ngram_range=(1,2), max_features=1500, dtype=np.float32)
        Xtr.append(tf.fit_transform(tr["texto_anon"])); Xte.append(tf.transform(te["texto_anon"]))
        nombres += [f"txt:{t}" for t in tf.get_feature_names_out()]
    return hstack(Xtr).tocsr(), hstack(Xte).tocsr(), nombres

resultados = []
def evaluar(nombre, modelo, Xtr, Xte, y_tr, y_te):
    t0 = time.time(); modelo.fit(Xtr, y_tr); proba = modelo.predict_proba(Xte)
    clases = modelo.classes_; pred = clases[proba.argmax(1)]; conf = proba.max(1)
    r = {"modelo": nombre, "acierto": accuracy_score(y_te, pred), "F1 macro": f1_score(y_te, pred, average="macro"),
         "top-3": top_k_accuracy_score(y_te, proba, k=3, labels=clases)}
    for u in (0.9, 0.8):
        m = conf >= u; r[f"lineas con conf>={u}"] = m.mean(); r[f"acierto en ellas (>={u})"] = accuracy_score(y_te[m], pred[m]) if m.any() else np.nan
    r["segundos"] = round(time.time()-t0); resultados.append(r)
    print(f"{nombre}: acierto {r['acierto']:.1%} | F1 macro {r['F1 macro']:.3f} | top-3 {r['top-3']:.1%} | "
          f"con confianza >= 0.9: {r['lineas con conf>=0.9']:.0%} de las lineas, acierto {r['acierto en ellas (>=0.9)']:.1%} | {r['segundos']} s")
    return pred, conf, proba

Xtr, Xte, nombres = matrices(tr, te)
print(f"Matriz de entrenamiento: {Xtr.shape[0]:,} lineas x {Xtr.shape[1]:,} variables")

## 3.3 · Modelo A — Regresión logística multinomial

**Supuesto:** cada cuenta se separa de las demás con una frontera **lineal** en el espacio de variables. Cada variable suma o resta puntos a favor de cada cuenta, de forma independiente. Es el modelo más interpretable —los coeficientes se leen— y el que mejor suele aprovechar miles de variables dispersas (columnas 0/1 y palabras), porque en ese espacio la mayoría de las clases sí son linealmente separables.

In [ ]:
modelo_A = LogisticRegression(max_iter=3000, C=1.0)
predA, confA, probaA = evaluar("A · Logistica multinomial", modelo_A, Xtr, Xte, y_tr, y_te)
clases_A = modelo_A.classes_

## 3.4 · Modelo B — Bosque aleatorio

**Supuesto opuesto:** no hay frontera lineal; la cuenta se decide por **reglas anidadas** (si es crédito y el documento tiene más de 3 líneas y el monto está en tal rango…), aprendidas por muchos árboles que votan. Captura interacciones que la logística no puede, pero en cada corte mira solo un puñado de variables al azar, lo que lo castiga cuando hay miles de columnas dispersas.

Se limita la profundidad y el tamaño mínimo de hoja para que quepa en memoria: con sesenta clases cada hoja almacena sesenta probabilidades.

In [ ]:
modelo_B = RandomForestClassifier(n_estimators=120, min_samples_leaf=5, max_depth=30, n_jobs=-1, random_state=SEMILLA, class_weight="balanced_subsample")
predB, confB, probaB = evaluar("B · Bosque aleatorio", modelo_B, Xtr, Xte, y_tr, y_te)

## 3.5 · Comparación

In [ ]:
te["predA"], te["confA"], te["predB"], te["confB"] = predA, confA, predB, confB
te["ok_B2"] = (te.B2==te.objetivo).astype(int); te["ok_A"] = (te.predA==te.objetivo).astype(int); te["ok_B"] = (te.predB==te.objetivo).astype(int)

comparacion = pd.DataFrame([{"modelo": "Regla B2 (tercero + tipo + naturaleza)", "acierto": te.ok_B2.mean(), "F1 macro": f1_score(y_te, te.B2, average="macro"), "top-3": np.nan}] + resultados[:2])
comparacion["error (1 - acierto)"] = 1 - comparacion["acierto"]
display(comparacion.set_index("modelo")[["acierto","error (1 - acierto)","F1 macro","top-3","lineas con conf>=0.9","acierto en ellas (>=0.9)"]])
mejor = comparacion.iloc[1:].sort_values("error (1 - acierto)").iloc[0]
print(f"Modelo con menor error: {mejor['modelo']} ({mejor['error (1 - acierto)']:.1%} de error frente a {comparacion.iloc[2]['error (1 - acierto)']:.1%} del otro y {comparacion.iloc[0]['error (1 - acierto)']:.1%} de la regla).")

por_emp = te.groupby(["software","empresa"])[["ok_B2","ok_A","ok_B"]].mean(); por_emp.columns = ["regla B2","logistica A","bosque B"]
por_gr  = te.groupby("grupo_puc")[["ok_B2","ok_A","ok_B"]].mean();            por_gr.columns  = ["regla B2","logistica A","bosque B"]
por_nue = te.groupby("tercero_nuevo")[["ok_B2","ok_A","ok_B"]].mean();        por_nue.columns = ["regla B2","logistica A","bosque B"]; por_nue.index = ["tercero conocido","tercero NUEVO en 2026"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
por_emp.plot(kind="bar", ax=axes[0], color=[GRIS, VERDE, ROJO]); axes[0].set_title("Por empresa"); axes[0].set_ylim(0,1)
axes[0].set_xticklabels([f"{s}\n{e}" for s,e in por_emp.index], rotation=0, fontsize=8)
por_gr.plot(kind="bar", ax=axes[1], color=[GRIS, VERDE, ROJO], legend=False); axes[1].set_title("Por grupo del PUC"); axes[1].set_ylim(0,1); axes[1].tick_params(axis="x", rotation=0)
por_nue.plot(kind="bar", ax=axes[2], color=[GRIS, VERDE, ROJO], legend=False); axes[2].set_title("Segun si el tercero ya existia"); axes[2].set_ylim(0,1); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
display(por_nue)
print(f"En gastos (grupo 5): regla {por_gr.loc['5','regla B2']:.1%} -> logistica {por_gr.loc['5','logistica A']:.1%}.")

**Lectura.** Los dos modelos superan con claridad a la regla, y **la regresión logística multinomial es el modelo de menor error**: la mitad del error del bosque aleatorio y menos de un tercio del de la regla. Gana en todas las métricas y en todas las empresas. La razón está en los supuestos: con miles de columnas dispersas —el tercero en one-hot y las palabras del texto— la frontera lineal usa toda la información a la vez, mientras el bosque, que en cada corte muestrea unas pocas variables, ve casi siempre ceros. Además la confianza del bosque está peor calibrada: pocas líneas alcanzan 0,9.

Lo que más vale del modelo está en la tercera gráfica: **sobre los terceros nuevos, donde la regla no puede hacer nada, la logística conserva casi todo su acierto.** No depende de memorizar quién es el tercero; aprendió cómo se codifica una línea.

## 3.6 · Qué aporta cada fuente de información

Se reentrena la logística quitando una fuente a la vez. La diferencia con el modelo completo es lo que esa fuente vale.

In [ ]:
for con_t, con_x, nom in [(True, False, "A · sin texto"), (False, True, "A · sin identidad del tercero"), (False, False, "A · solo contexto del comprobante (sin tercero ni texto)")]:
    a, b, _ = matrices(tr, te, con_tercero=con_t, con_texto=con_x)
    evaluar(nom, LogisticRegression(max_iter=3000, C=1.0), a, b, y_tr, y_te)
abl = pd.DataFrame(resultados).set_index("modelo")[["acierto","F1 macro","top-3"]]
abl["perdida vs completo"] = abl["acierto"] - abl.loc["A · Logistica multinomial","acierto"]
display(abl)

**Lectura.** El **texto** es la fuente que más aporta; la **identidad del tercero**, sorprendentemente poco — el modelo la sustituye con el contexto del comprobante y con qué tan habitual es el tercero. Y el modelo que solo ve la estructura del comprobante, sin saber quién es el tercero ni qué dice la descripción, **todavía supera a la regla B2**. Eso valida el Hallazgo 1 del diagnóstico con el modelo en la mano.

Para la firma, la consecuencia es concreta: el export que se le pida a un software **tiene que traer la descripción del movimiento**; sin ella se pierden cerca de diez puntos de acierto.

## 3.7 · ¿Viaja el modelo entre softwares?

Se entrena solo con las empresas de un software y se prueba con las del otro, usando únicamente las variables comunes.

In [ ]:
filas = []
for origen, destino in [("LOGGRO","SIIGO"), ("SIIGO","LOGGRO")]:
    a = lb[(lb.software==origen)&(lb.anio==2025)]; b = lb[(lb.software==destino)&(lb.anio==2026)]; propio = lb[(lb.software==destino)&(lb.anio==2025)]
    def X(d): return np.hstack([d[NUMERICAS].values.astype(np.float32), (d["naturaleza"]=="D").values.astype(np.float32)[:,None]])
    sc = StandardScaler().fit(X(a));       m  = LogisticRegression(max_iter=2000).fit(sc.transform(X(a)), a.objetivo)
    sc2 = StandardScaler().fit(X(propio)); m2 = LogisticRegression(max_iter=2000).fit(sc2.transform(X(propio)), propio.objetivo)
    filas.append({"entrenado en": origen, "probado en": destino,
                  "acierto cruzado": accuracy_score(b.objetivo, m.predict(sc.transform(X(b)))),
                  "acierto entrenando en el mismo software": accuracy_score(b.objetivo, m2.predict(sc2.transform(X(b))))})
display(pd.DataFrame(filas).set_index(["entrenado en","probado en"]))

**Lectura.** Incluso con solo la estructura del comprobante, un modelo entrenado en un software acierta una fracción de lo que acierta dentro del mismo. **La forma de codificar no viaja entre sistemas** — ni siquiera la forma en que cada software arma un comprobante es la misma. Consecuencia para la recomendación: el modelo se entrena **por sistema y por cartera**, y no se puede prometer que sirva para otra firma sin reentrenarlo.

## 3.8 · Modelo no supervisado — ¿Qué líneas son raras para su propia cuenta?

Hasta aquí los modelos aprendieron de la etiqueta. Ahora se tapa la etiqueta y se pregunta otra cosa: **dentro de cada cuenta, ¿qué líneas se apartan del patrón de sus pares?** Es lo que un auditor hace al revisar un mayor: no compara un gasto de nómina con un pago a proveedor, compara cada gasto de nómina con los demás gastos de nómina.

Se usa un *Isolation Forest* por cuenta. La intuición: una línea normal necesita muchos cortes aleatorios para quedar aislada; una atípica, muy pocos. La rareza se mide contra su propia cuenta, y se marca el 3 % más raro de cada una, para que el criterio sea comparable entre cuentas grandes y pequeñas.

In [ ]:
COLS_ISO = ["log_monto","monto_redondo","dia_semana","fin_de_mes","cierre_dic","n_lineas","prop_del_cbte","es_linea_mayor","n_debitos","n_creditos","tercero_freq"]
te["rareza"] = np.nan
for cta, grp in tr.groupby("cuenta4"):
    idx = te.index[te.cuenta4 == cta]
    if len(grp) < 150 or len(idx) == 0: continue
    sc = StandardScaler().fit(grp[COLS_ISO])
    iso = IsolationForest(n_estimators=100, contamination="auto", random_state=SEMILLA).fit(sc.transform(grp[COLS_ISO]))
    te.loc[idx, "rareza"] = -iso.score_samples(sc.transform(te.loc[idx, COLS_ISO]))
te["atipico"] = 0
for cta, gr in te[te.rareza.notna()].groupby("cuenta4"):
    te.loc[gr.index[gr.rareza >= gr.rareza.quantile(0.97)], "atipico"] = 1

print(f"Lineas evaluadas: {te.rareza.notna().mean():.0%} | marcadas como atipicas para su cuenta: {te.atipico.mean():.1%}")
cruce = te.groupby("atipico").agg(lineas=("monto","size"), acierto_logistica=("ok_A","mean"))
lg = te[te.es_manual >= 0]; cruce["prop_asientos_manuales (Loggro)"] = lg.groupby("atipico")["es_manual"].mean()
cruce.index = ["normal","atipica"]; display(cruce)

**Lectura.** Los dos modelos, que nunca se hablaron, **señalan el mismo sitio**: en las líneas que el no supervisado marca como raras para su cuenta, la logística acierta bastante menos, y entre ellas hay más asientos manuales. Lo que es raro para su cuenta es también difícil de codificar y más probable que sea manual. Eso convierte la rareza en un criterio de priorización de revisión que el clasificador solo no habría dado.

Una advertencia que hay que decir en voz alta: **atípico no es erróneo.** Es una línea que hay que preguntar.

## 3.10 · Un modelo especializado en cuentas de resultado

El nivel diagnóstico y la comparación por grupo dejaron claro que la ambigüedad está en **ingresos, gastos y costos** (grupos 4 a 7): el activo y el pasivo se codifican casi solos porque salen del documento. La pregunta natural es si un modelo entrenado **únicamente** con esas líneas lo hace mejor que el general sobre ellas.

In [ ]:
res_mask = lb.grupo_puc.isin(["4","5","6","7"])
tr_res, te_res = tr[tr.grupo_puc.isin(["4","5","6","7"])], te[te.grupo_puc.isin(["4","5","6","7"])]
print(f"Lineas de resultado: {res_mask.sum():,} ({res_mask.mean():.0%} del libro) | entrenamiento {len(tr_res):,} | prueba {len(te_res):,} | clases {tr_res.objetivo.nunique()}")

Xtr_r, Xte_r, _ = matrices(tr_res, te_res)
modelo_R = LogisticRegression(max_iter=3000, C=1.0).fit(Xtr_r, tr_res.objetivo)
proba_r = modelo_R.predict_proba(Xte_r); pred_r = modelo_R.classes_[proba_r.argmax(1)]
m_res = te.grupo_puc.isin(["4","5","6","7"]).values
esp = pd.DataFrame([
    {"modelo": "General (60 clases), evaluado solo en lineas de resultado", "acierto": accuracy_score(y_te[m_res], predA[m_res]),
     "F1 macro": f1_score(y_te[m_res], predA[m_res], average="macro"), "top-3": np.nan},
    {"modelo": "Especializado en resultado (entrenado solo con grupos 4-7)", "acierto": accuracy_score(te_res.objetivo, pred_r),
     "F1 macro": f1_score(te_res.objetivo, pred_r, average="macro"), "top-3": top_k_accuracy_score(te_res.objetivo, proba_r, k=3, labels=modelo_R.classes_)},
]).set_index("modelo"); esp["error"] = 1 - esp["acierto"]
display(esp[["acierto","error","F1 macro","top-3"]])
print(f"Ganancia del especialista sobre las lineas de resultado: {esp.iloc[1]['acierto']-esp.iloc[0]['acierto']:+.1%} puntos de acierto.")

**Lectura.** El especialista gana varios puntos justo donde está el trabajo humano. Parte de esa ventaja es legítima —aprende con más detalle la frontera entre cuentas de gasto parecidas— y parte es estructural: **como nunca ve activos ni pasivos, no puede confundir un gasto con uno de ellos**; elige entre menos de treinta cuentas en vez de sesenta. Conviene decirlo así en la sustentación.

Para el protocolo, la combinación es natural: el modelo general clasifica la línea y decide si es de resultado; si lo es, el especialista afina la sugerencia. En el piloto se mide si esa segunda opinión reduce el reproceso.

## 3.9 · Elección del modelo y qué se pierde

**Modelo elegido: la regresión logística multinomial** (general, con su versión especializada en resultado como refuerzo), por el criterio más simple y defendible — es el de **menor error** en datos que nunca vio (2026) — y por tres razones más: mejor F1 macro (las cuentas raras también se aciertan), mejor top-3 (la cuenta correcta está entre las tres sugeridas el 96 % de las veces, que es lo que importa para un asistente), confianza bien calibrada (cuando dice 0,9, acierta 98 %) y coeficientes que se pueden leer. Lo que se pierde frente al bosque es la capacidad de capturar interacciones no lineales entre variables; en estos datos esa capacidad no compensó, y el bosque se queda como contraste que muestra que el resultado no depende del algoritmo elegido. El *Isolation Forest* no compite con ninguno de los dos: responde una pregunta distinta y se usa **junto** con la logística en el nivel prescriptivo.

---
# 4 · NIVEL PRESCRIPTIVO · ¿Qué hacemos?

Predecir no es decidir. El modelo entrega, para cada línea, una cuenta sugerida y una confianza; el no supervisado entrega una rareza. Lo que el auxiliar nuevo necesita es un **protocolo**: cuándo seguir la sugerencia, cuándo elegir entre varias y cuándo parar y consultar. Eso es lo que se construye aquí. Como beneficio secundario se cuantifica cuántas horas ahorra el equipo si las líneas de mayor confianza se automatizan; ese cálculo exige supuestos de tiempos que no están en los datos y que el socio debe validar.

## 4.1 · El umbral es la decisión

In [ ]:
umbrales = np.linspace(0.3, 0.99, 30)
cobertura = [(confA>=u).mean() for u in umbrales]; acierto_u = [accuracy_score(y_te[confA>=u], predA[confA>=u]) for u in umbrales]
fig, ax = plt.subplots()
ax.plot(umbrales, cobertura, "o-", color=GRIS, label="cobertura: lineas que se automatizarian"); ax.plot(umbrales, acierto_u, "s-", color=VERDE, label="acierto en esas lineas")
ax.axvline(0.9, ls="--", color=ROJO); ax.set_xlabel("umbral de confianza"); ax.set_ylim(0, 1.02); ax.legend(); ax.set_title("Cuanto se automatiza y con que acierto, segun donde se ponga el umbral"); plt.show()
i = np.argmin(np.abs(umbrales-0.9)); print(f"Con umbral 0.9: se automatiza el {cobertura[i]:.0%} de las lineas con {acierto_u[i]:.1%} de acierto.")

**Lectura.** No hay un umbral "correcto": cada punto de la curva es una combinación de cuánto se confía en la sugerencia y cuántos errores se aceptan. Se propone **0,9** como frontera de la zona verde porque a partir de ahí el acierto supera el 98 % —el auxiliar que siga la sugerencia codifica como el equipo— y todavía cubre dos de cada tres líneas. El socio puede moverlo.

## 4.2 · El protocolo de tres zonas para el auxiliar nuevo

Cada línea cae en una zona según la confianza del asistente y la rareza dentro de su cuenta. La zona le dice al auxiliar qué hacer:

| Zona | Criterio | Qué hace el auxiliar nuevo |
|---|---|---|
| 1 · Sigue | confianza ≥ 0,9 y no atípica | Registra la cuenta sugerida. Si su criterio dice otra cosa, la anota y sigue: se revisa después por muestreo |
| 2 · Elige entre tres | confianza entre 0,5 y 0,9 y no atípica | El asistente muestra las tres cuentas más probables; el auxiliar decide con su criterio |
| 3 · Consulta | atípica para su cuenta, o confianza < 0,5 | No registra sin consultar a un senior. Es la lista de lo que un nuevo *debe* preguntar |

Para la firma, la misma zona 1 es la candidata a **automatizarse** cuando el auxiliar ya no es nuevo; por eso se calcula también cuántas horas ahorraría.

In [ ]:
# ---- SUPUESTOS DEL GRUPO: cuanto tarda una persona por linea en cada modalidad, y cuanto vale su hora. Ajustar con el socio.
MIN_MANUAL   = 1.0    # minutos por linea codificada a mano, como hoy
MIN_ASISTIDA = 0.4    # minutos por linea cuando el sistema sugiere tres cuentas
MIN_REVISION = 3.0    # minutos por linea que se revisa a fondo
MIN_MUESTREO = 0.1    # minutos promedio por linea automatizada (control por muestreo del 10 %)
TARIFA_HORA  = 30000  # COP por hora del auxiliar contable
UMBRAL_AUTO, UMBRAL_ASIST = 0.9, 0.5

def zona(r):
    if r.atipico == 1:          return "3 · Consulta (atipica para su cuenta)"
    if r.confA >= UMBRAL_AUTO:  return "1 · Sigue la sugerencia"
    if r.confA >= UMBRAL_ASIST: return "2 · Elige entre tres"
    return "3 · Consulta (baja confianza)"
te["zona"] = te.apply(zona, axis=1)
meses = te["mes"].nunique()
te["ok_top3"] = [o in set(clases_A[np.argsort(-p)[:3]]) for o, p in zip(te.objetivo, probaA)]
z = te.groupby("zona").agg(lineas=("monto","size"), acierto_top1=("ok_A","mean"), acierto_top3=("ok_top3","mean")).reset_index()
z["proporcion"] = z.lineas/len(te); z["lineas_por_mes"] = z.lineas/meses
minutos = {"1 · Sigue la sugerencia": MIN_MUESTREO, "2 · Elige entre tres": MIN_ASISTIDA, "3 · Consulta (atipica para su cuenta)": MIN_REVISION, "3 · Consulta (baja confianza)": MIN_REVISION}
z["horas_por_mes_propuesto"] = z.zona.map(minutos)*z.lineas_por_mes/60
z["horas_por_mes_hoy"] = MIN_MANUAL*z.lineas_por_mes/60
display(z.set_index("zona"))

horas_hoy = z.horas_por_mes_hoy.sum(); horas_prop = z.horas_por_mes_propuesto.sum()
auto = z[z.zona=="1 · Sigue la sugerencia"].iloc[0]; revisar = z[z.zona.str.startswith("3")]; elige = z[z.zona.str.startswith("2")].iloc[0]
print(f"Un auxiliar nuevo que siga el protocolo coincide con el equipo experimentado en el {(te.ok_A[te.zona.str.startswith('1')].sum() + te.ok_top3[te.zona.str.startswith('2')].sum()) / len(te[~te.zona.str.startswith('3')]):.1%} de las lineas que registra sin consultar (zonas 1 y 2).")
print("\n--- Beneficio secundario: horas del equipo si la zona 1 se automatiza (supuestos de tiempos) ---")
print(f"Lineas por mes en las {te.empresa.nunique()} empresas: {len(te)/meses:,.0f}")
print(f"Horas/mes codificando todo a mano (hoy): {horas_hoy:,.0f}  ->  con el esquema propuesto: {horas_prop:,.0f}  ->  ahorro {horas_hoy-horas_prop:,.0f} h/mes "
      f"(~${(horas_hoy-horas_prop)*TARIFA_HORA:,.0f} COP/mes con la tarifa supuesta)")
print(f"Lineas de la zona 1 con error: ~{auto.lineas_por_mes*(1-auto.acierto_top1):,.0f} por mes, que el muestreo debe atrapar.")
print(f"Lineas que el auxiliar nuevo debe consultar: {revisar.lineas_por_mes.sum():,.0f} por mes ({revisar.proporcion.sum():.0%}).")
zpe = te.groupby(["empresa","zona"]).size().unstack(fill_value=0); zpe["% zona 1"] = zpe["1 · Sigue la sugerencia"]/zpe.sum(axis=1); display(zpe)

## 4.3 · La recomendación

In [ ]:
print(f'''
Al socio de la firma:

Cada auxiliar que entra a la firma tarda meses en codificar como el equipo. Entrenamos un modelo con la codificacion que el
equipo hizo en 2025 sobre {len(lb):,} movimientos de {lb.empresa.nunique()} empresas y dos softwares, y lo probamos sobre 2026.
Su sugerencia coincide con la del equipo en el {te.ok_A.mean():.0%} de las lineas; la cuenta correcta esta entre sus tres
sugerencias en el {resultados[0]["top-3"]:.0%}; y conserva el {por_nue.loc["tercero NUEVO en 2026","logistica A"]:.0%} de acierto con terceros
que nadie habia visto, que es donde un nuevo mas se equivoca.

Recomendamos adoptar el asistente como referencia obligatoria en los primeros tres meses de cada auxiliar, con este protocolo:
  1. Sigue la sugerencia en el {auto.proporcion:.0%} de las lineas (confianza >= {UMBRAL_AUTO}): acierto {auto.acierto_top1:.1%}.
  2. Elige entre tres en el {elige.proporcion:.0%}: la cuenta correcta esta entre ellas el {elige.acierto_top3:.0%} de las veces.
  3. Consulta antes de registrar en el {revisar.proporcion.sum():.0%}: lineas raras para su cuenta o donde el asistente duda.

Como se mide si funciona: tasa de reproceso en revision de los auxiliares nuevos, antes y despues del asistente.
Beneficio secundario, con los tiempos supuestos: si la zona 1 se automatiza para el equipo experimentado, el trabajo de
codificacion pasa de {horas_hoy:,.0f} a {horas_prop:,.0f} horas al mes en estas seis empresas.

Lo que pedimos: aprobar un piloto de tres meses con el proximo auxiliar que entre, en una empresa de cada software.
''')

---
# 5 · Límites — dicho por nosotros antes de que lo pregunten

1. **Seis empresas de una sola firma.** No representan el tejido empresarial ni otras prácticas contables. El modelo es de esta cartera.
2. **No viaja entre sistemas.** La sección 3.7 lo mide: entrenado en un software, en el otro no sirve. Cualquier despliegue exige reentrenar por sistema, y probablemente por empresa.
3. **Los datos no traen quién registró ni cuándo se creó el asiento.** El detector de anomalías trabaja con la estructura del movimiento, no con el comportamiento de quien lo hizo; una prueba de asientos de diario completa necesitaría esas dos columnas.
4. **2026 no está cerrado.** Los últimos meses pueden cambiar con ajustes de cierre; los resultados sobre 2026 son sobre la contabilidad tal como estaba al momento del export.
5. **Los tiempos por línea son supuestos.** El ahorro en horas y pesos depende de ellos y se validan con el socio antes de decidir; el notebook los tiene como parámetros para recalcular.
6. **Una de las empresas es una clínica** y sus terceros son pacientes. Están anonimizados y el texto describe el servicio, no información clínica, pero es una fuente que exige más cuidado que las demás.
7. **"Atípico" no significa "error"** y "confianza alta" no significa "correcto": el 98 % de acierto en la zona 1 sigue dejando líneas mal codificadas, y por eso lleva muestreo, no fe.
8. **El asistente reproduce cómo codifica el equipo, no cómo debería codificar.** Si en 2025 una clase de movimiento se registró mal de forma sistemática, el asistente lo aprendió. Es una referencia de la práctica de la firma, no del PUC.

### Siguientes pasos

- Piloto de tres meses con el próximo auxiliar que entre, en una empresa de cada software, midiendo reproceso antes y después.
- Incorporar usuario y fecha de creación al export, para pasar de rareza estructural a rareza de comportamiento.
- Reentrenar cada trimestre: la deriva de la sección 2.4 es real.

---
### Anexo · Diccionario de columnas de `libro_comun_anon.csv`

| Columna | Qué es |
|---|---|
| `empresa`, `software` | Código anónimo de la empresa (E01…) y software de origen |
| `fecha`, `anio`, `mes`, `dia_semana`, `fin_semana`, `fin_de_mes`, `cierre_dic` | Fecha del movimiento y derivadas de calendario |
| `comprobante_id` | Identificador único del comprobante (empresa + año + número) |
| `tipo_doc` | Tipo de documento según el software (categorías estándar en Loggro; prefijo del comprobante en Siigo) |
| `tercero` | Código anónimo del tercero, consistente entre empresas |
| `tipo_persona` | Natural / Jurídica (solo Loggro lo exporta) |
| `centro` | Centro de costo |
| `texto_anon` | Descripción del movimiento, con nombres de terceros reemplazados por `<T>` |
| `es_manual` | 1 = asiento manual, 0 = automático, −1 = el software no lo informa |
| `debito`, `credito`, `monto`, `log_monto`, `monto_redondo`, `naturaleza` | Cifras escaladas por una constante; naturaleza D/C |
| `cuenta`, `cuenta4`, `grupo_puc` | Cuenta original, a cuatro dígitos, y su grupo (primer dígito) |
| `objetivo` | Variable objetivo: `cuenta4` si tiene ≥ 100 movimientos, si no `OTRAS` |
| `anulado` | Marca de comprobante anulado |